# NDI duplicate-day diagnostic (read-only)

The first six code cells are copied unchanged from `NDI_v1_3_forward.ipynb` (data loading only). The last cell lists every duplicated date, the wellness reports behind it, and whether local-time dating removes it. It writes only to `results_diagnostics/`; nothing in `results_final`, `results_v1.2` or `results_v1.3` is touched.

Runtime > Run all. About 5-10 minutes.

In [ ]:
# ================================================================
# CELL 1.0  CONFIGURATION  (edit paths to match your Drive)
# Brief Section 6: all parameters in one place, fixed seed
# ================================================================

#  Google Drive paths
PIPELINE_MODULE   = "/content/drive/MyDrive/ndi_project/ndi_pipeline.py"
DEPRESJON_DIR     = "/content/drive/MyDrive/ndi_project/depresjon-dataset/data"
PMDATA_DIR        = "/content/drive/MyDrive/ndi_project/pmdata"
PMDATA_OVERVIEW   = "/content/drive/MyDrive/ndi_project/pmdata/participant-overview.xlsx"
RESULTS_DIR       = "/content/drive/MyDrive/ndi_project/results"

#  Reproducibility seed (Brief Section 6)
RANDOM_SEED = 42

#  Mount Drive
from google.colab import drive
drive.mount("/content/drive")

import os, sys
os.makedirs(RESULTS_DIR, exist_ok=True)

# Add pipeline module to path
sys.path.insert(0, os.path.dirname(PIPELINE_MODULE))

print("Drive mounted.")
print("DEPRESJON dir exists:", os.path.isdir(DEPRESJON_DIR))
print("PMData    dir exists:", os.path.isdir(PMDATA_DIR))
print("Pipeline  exists    :", os.path.isfile(PIPELINE_MODULE))


In [ ]:
# Cell 1.1 --- results-capture helper (moved here so item 1 / item 3 / item 4 / item 7
#     calls earlier in the notebook can all use it) ---
import json, datetime

RESULTS_FINAL_DIR = os.path.join(os.path.dirname(RESULTS_DIR), "results_final")
os.makedirs(RESULTS_FINAL_DIR, exist_ok=True)
MANIFEST_PATH = os.path.join(RESULTS_FINAL_DIR, "manifest.json")
_manifest = []

def save_result(item_no, description, obj, filename, cell_label):
    path = os.path.join(RESULTS_FINAL_DIR, filename)
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=False)
    else:
        with open(path, "w") as f:
            json.dump(obj, f, indent=2, default=str)
    _manifest.append(dict(item=item_no, description=description, cell=cell_label,
                           file=filename, saved_at=datetime.datetime.now().isoformat()))
    with open(MANIFEST_PATH, "w") as f:
        json.dump(_manifest, f, indent=2)
    print(f"[item {item_no}] saved -> {filename}")

In [ ]:
import shutil, json, os, numpy as np

_parent = os.path.dirname(RESULTS_DIR)
FROZEN_V1_DIR = os.path.join(_parent, "results_final_v1_frozen_backup")

if not os.path.exists(FROZEN_V1_DIR):
    _existing_manifest = json.load(open(os.path.join(RESULTS_FINAL_DIR, "manifest.json")))
    print(f"Current manifest.json has {len(_existing_manifest)} entries, items present: "
          f"{sorted(set(m['item'] for m in _existing_manifest))}")
    print("Confirm this looks like your complete v1 record before proceeding.")
    shutil.copytree(RESULTS_FINAL_DIR, FROZEN_V1_DIR)
    print(f"Backed up to {FROZEN_V1_DIR}")
else:
    print(f"Frozen backup already exists at {FROZEN_V1_DIR} -- not overwriting.")

RESULTS_FINAL_DIR = os.path.join(_parent, "results_regen_scratch")   # this session writes here only
os.makedirs(RESULTS_FINAL_DIR, exist_ok=True)
MANIFEST_PATH = os.path.join(RESULTS_FINAL_DIR, "manifest.json")
_manifest = []

def benjamini_hochberg(pvals):   # identical to the CELL 14.9 definition -- redefined here
    pvals = np.asarray(pvals, dtype=float); n = len(pvals)
    order = np.argsort(pvals); q_raw = pvals[order] * n / (np.arange(n) + 1)
    q_sorted = np.minimum.accumulate(q_raw[::-1])[::-1]
    q = np.empty(n); q[order] = np.clip(q_sorted, 0, 1); return q

print("Frozen v1 backup:", FROZEN_V1_DIR, "| session writes ->", RESULTS_FINAL_DIR)

In [ ]:
# ================================================================
# CELL 2  INSTALL DEPENDENCIES + IMPORTS
# ================================================================
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["xgboost", "shap"]:
    install(pkg)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.tri as tri
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings("ignore")
np.random.seed(RANDOM_SEED)

import importlib.util, sys as _sys

spec = importlib.util.spec_from_file_location("ndi_pipeline", PIPELINE_MODULE)
ndi  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ndi)

print("ndi_pipeline loaded. Components available:")
for fn in ["neutrosophic_decompose","blocked_tscv","run_baselines",
           "lambda_grid_search","shap_attribution"]:
    print(f"  ndi.{fn}: {'OK' if hasattr(ndi, fn) else 'MISSING'}")


In [ ]:
# ================================================================
# CELL 5 -- LOAD PMDATA DATASET
# Actual structure:
#   pmdata/p01/fitbit/heart_rate.json
#   pmdata/p01/fitbit/resting_heart_rate.json
#   pmdata/p01/fitbit/sleep.json
#   pmdata/p01/fitbit/sleep_score.csv
#   pmdata/p01/fitbit/steps.json
#   pmdata/p01/fitbit/lightly_active_minutes.json
#   pmdata/p01/fitbit/moderately_active_minutes.json
#   pmdata/p01/fitbit/very_active_minutes.json
#   pmdata/p01/fitbit/sedentary_minutes.json
#   pmdata/p01/pmsys/wellness.csv    <- daily mood/wellness self-report
#   pmdata/p01/pmsys/srpe.csv
#   pmdata/participant-overview.xlsx <- max HR, age, gender per participant
# ================================================================
import os, glob, json as _json
import pandas as pd
import numpy as np
import openpyxl  # for reading xlsx

#  Load participant overview
try:
    overview = pd.read_excel(PMDATA_OVERVIEW, header=1)
    overview.columns = [str(c).strip().lower().replace(" ","_")
                        for c in overview.columns]
    overview = overview.dropna(subset=[overview.columns[0]])
    print("participant-overview columns:", list(overview.columns))
    print(overview.head(3))
except Exception as e:
    print(f"Could not load participant-overview: {e}")
    overview = pd.DataFrame()

#  Helper: load one JSON Fitbit file
def load_fitbit_json(path):
    """Load a Fitbit JSON file. Returns list of {dateTime, value} dicts."""
    try:
        with open(path) as f:
            data = _json.load(f)
        if isinstance(data, list):
            return data
        return []
    except Exception:
        return []

def fitbit_to_daily(entries, value_key="value", agg="mean"):
    """Aggregate minute/entry-level Fitbit data to daily values."""
    rows = {}
    for entry in entries:
        date = str(entry.get("dateTime",""))[:10]
        val  = entry.get(value_key, entry.get("value", None))
        if date and val is not None:
            try:
                val = float(val)
                rows.setdefault(date, []).append(val)
            except (ValueError, TypeError):
                pass
    if agg == "mean":
        return {d: np.mean(v) for d,v in rows.items()}
    elif agg == "sum":
        return {d: np.sum(v) for d,v in rows.items()}
    elif agg == "max":
        return {d: np.max(v) for d,v in rows.items()}
    return rows

#  Helper: load resting heart rate JSON
def load_resting_hr(path):
    """resting_heart_rate.json has nested value.value structure."""
    entries = load_fitbit_json(path)
    rows = {}
    for entry in entries:
        date = str(entry.get("dateTime",""))[:10]
        val  = entry.get("value",{})
        if isinstance(val, dict):
            rhr = val.get("value", val.get("restingHeartRate", None))
        else:
            rhr = val
        if date and rhr is not None:
            try:
                rows[date] = float(rhr)
            except (ValueError, TypeError):
                pass
    return rows

#  Helper: load sleep.json
def load_sleep(path):
    """Extract sleep efficiency and minutes asleep from sleep.json."""
    entries = load_fitbit_json(path)
    eff_rows = {}; min_rows = {}
    for entry in entries:
        # Each entry is one sleep session
        date = str(entry.get("dateOfSleep", entry.get("dateTime","")))[:10]
        eff  = entry.get("efficiency", None)
        mins = entry.get("minutesAsleep", None)
        if date and eff is not None:
            try:
                eff_rows[date] = float(eff)
            except (ValueError, TypeError):
                pass
        if date and mins is not None:
            try:
                min_rows[date] = float(mins)
            except (ValueError, TypeError):
                pass
    return eff_rows, min_rows

#  Helper: load sleep_score.csv
def load_sleep_score(path):
    """sleep_score.csv: timestamp, overall_score, composition_score, ..."""
    try:
        df = pd.read_csv(path)
        df.columns = [c.strip().lower() for c in df.columns]
        # Timestamp column
        ts_col = next((c for c in df.columns if "timestamp" in c or "date" in c), None)
        score_col = next((c for c in df.columns if "overall" in c or "score" in c), None)
        if ts_col and score_col:
            df["date"] = pd.to_datetime(df[ts_col]).dt.strftime("%Y-%m-%d")
            return dict(zip(df["date"], pd.to_numeric(df[score_col], errors="coerce")))
    except Exception:
        pass
    return {}

#  Helper: load wellness.csv
def load_wellness(path):
    """
    pmsys/wellness.csv: daily self-report.
    Columns typically: date, fatigue, mood, readiness, sleep_duration,
                       sleep_quality, stress, soreness
    """
    try:
        df = pd.read_csv(path)
        df.columns = [c.strip().lower().replace(" ","_") for c in df.columns]
        if "date" not in df.columns:
            return pd.DataFrame()
        df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
        return df
    except Exception:
        return pd.DataFrame()

#  Helper: load googledocs/reporting.csv
def load_reporting(path):
    """
    reporting.csv: date, timestamp, meals, weight, glasses_of_fluid, alcohol
    """
    try:
        df = pd.read_csv(path)
        df.columns = [c.strip().lower().replace(" ","_") for c in df.columns]
        date_col = next((c for c in df.columns if "date" in c), None)
        if date_col:
            df["date"] = pd.to_datetime(df[date_col]).dt.strftime("%Y-%m-%d")
            return df
    except Exception:
        pass
    return pd.DataFrame()

#  Main loader
def load_pmdata_participant(pid_dir):
    """
    Load all data for one PMData participant.
    Returns a daily DataFrame with all available signals.
    """
    fitbit_dir    = os.path.join(pid_dir, "fitbit")
    pmsys_dir     = os.path.join(pid_dir, "pmsys")
    googledocs_dir= os.path.join(pid_dir, "googledocs")

    daily = {}  # date -> feature dict

    def add_series(series_dict, col_name):
        for date, val in series_dict.items():
            daily.setdefault(date, {})[col_name] = val

    # Resting heart rate (most reliable HR signal for daily level)
    rhr_path = os.path.join(fitbit_dir, "resting_heart_rate.json")
    if os.path.isfile(rhr_path):
        add_series(load_resting_hr(rhr_path), "resting_hr")

    # Steps
    steps_path = os.path.join(fitbit_dir, "steps.json")
    if os.path.isfile(steps_path):
        add_series(fitbit_to_daily(load_fitbit_json(steps_path), agg="sum"), "steps")

    # Activity minutes
    for fname, col in [
        ("lightly_active_minutes.json",    "lightly_active_min"),
        ("moderately_active_minutes.json", "moderately_active_min"),
        ("very_active_minutes.json",       "very_active_min"),
        ("sedentary_minutes.json",         "sedentary_min"),
    ]:
        p = os.path.join(fitbit_dir, fname)
        if os.path.isfile(p):
            add_series(fitbit_to_daily(load_fitbit_json(p), agg="sum"), col)

    # Sleep
    sleep_path = os.path.join(fitbit_dir, "sleep.json")
    if os.path.isfile(sleep_path):
        eff, mins = load_sleep(sleep_path)
        add_series(eff,  "sleep_efficiency")
        add_series(mins, "sleep_minutes")

    # Sleep score
    ss_path = os.path.join(fitbit_dir, "sleep_score.csv")
    if os.path.isfile(ss_path):
        add_series(load_sleep_score(ss_path), "sleep_score")

    if not daily:
        return pd.DataFrame()

    # Build base daily DataFrame
    df = pd.DataFrame.from_dict(daily, orient="index")
    df.index.name = "date"
    df = df.reset_index()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    # Active minutes total
    act_cols = [c for c in ["lightly_active_min","moderately_active_min","very_active_min"] if c in df.columns]
    if act_cols:
        df["active_minutes_total"] = df[act_cols].sum(axis=1)

    # Merge wellness
    w_path = os.path.join(pmsys_dir, "wellness.csv")
    if os.path.isfile(w_path):
        well = load_wellness(w_path)
        if not well.empty:
            well["date"] = pd.to_datetime(well["date"])
            df = df.merge(well, on="date", how="left")

    # Merge reporting (weight, fluid)
    r_path = os.path.join(googledocs_dir, "reporting.csv")
    if os.path.isfile(r_path):
        rep = load_reporting(r_path)
        if not rep.empty:
            rep["date"] = pd.to_datetime(rep["date"])
            rep_agg = rep.groupby("date").agg(
                weight         = ("weight", "mean"),
                glasses_of_fluid=("glasses_of_fluid","mean") if "glasses_of_fluid" in rep.columns else ("date","count"),
            ).reset_index()
            df = df.merge(rep_agg, on="date", how="left")

    df["date"] = df["date"].dt.strftime("%Y-%m-%d")
    return df


#  Load all participants
pmdata_participants = []
participant_dirs = sorted([
    os.path.join(PMDATA_DIR, d)
    for d in os.listdir(PMDATA_DIR)
    if os.path.isdir(os.path.join(PMDATA_DIR, d)) and d.startswith("p")
])
print(f"Found {len(participant_dirs)} participant directories")

for pdir in participant_dirs:
    pid = os.path.basename(pdir)
    try:
        daily = load_pmdata_participant(pdir)
        if len(daily) < 7:
            print(f"  SKIP {pid}: only {len(daily)} days")
            continue

        # Get max HR from overview
        max_hr = np.nan
        if not overview.empty:
            hr_col = next((c for c in overview.columns if "heart" in c and "max" in c), None)
            id_col = next((c for c in overview.columns if "participant" in c or "id" in c), None)
            if hr_col and id_col:
                row = overview[overview[id_col].astype(str).str.lower().str.strip() == pid.lower()]
                if not row.empty:
                    max_hr = float(row.iloc[0][hr_col]) if not pd.isna(row.iloc[0][hr_col]) else np.nan

        pmdata_participants.append(dict(
            pid       = pid,
            group     = "pmdata",
            is_depressed = 0,
            max_hr    = max_hr,
            n_days    = len(daily),
            days_df   = daily,
        ))
        cols_found = [c for c in daily.columns if c != "date"]
        print(f"  OK {pid}: {len(daily)} days  max_HR={max_hr:.0f}  signals={cols_found[:6]}")
    except Exception as e:
        print(f"  ERROR {pid}: {e}")

print(f"\nPMData: {len(pmdata_participants)} participants loaded")
if pmdata_participants:
    print(f"  Median days: {np.median([p['n_days'] for p in pmdata_participants]):.0f}")
    all_cols = set()
    for p in pmdata_participants:
        all_cols.update(p['days_df'].columns.tolist())
    print(f"  All available columns: {sorted(all_cols)}")


In [ ]:
# ================================================================
# CELL 6  PMData FEATURE ENGINEERING + WELLNESS MERGE
# Defines feature set, builds neutrosophic features, and crucially
# merges wellness scores as the external target column.
# ================================================================
import numpy as np
import pandas as pd

if not pmdata_participants:
    print("No PMData participants loaded. Check PMDATA_DIR.")
    pmdata_all = pd.DataFrame()
    PMDATA_FEATURE_NAMES = []
    PMDATA_DOMAIN_MAP    = {}
    PMDATA_WELLNESS      = []
else:
    # Collect all columns across participants
    sample_cols = set()
    for p in pmdata_participants:
        sample_cols.update(p["days_df"].columns.tolist())
    sample_cols.discard("date")

    PMDATA_PHYS = [c for c in [
        "resting_hr","steps","active_minutes_total",
        "lightly_active_min","moderately_active_min",
        "very_active_min","sedentary_min"
    ] if c in sample_cols]

    PMDATA_SLEEP = [c for c in [
        "sleep_efficiency","sleep_minutes","sleep_score"
    ] if c in sample_cols]

    # Wellness columns — these become the EXTERNAL TARGET for lambda search
    PMDATA_WELLNESS = [c for c in [
        "fatigue","mood","readiness","sleep_quality","stress","soreness"
    ] if c in sample_cols]

    PMDATA_FEATURE_NAMES = PMDATA_PHYS + PMDATA_SLEEP
    PMDATA_DOMAIN_MAP    = {"physiology": PMDATA_PHYS, "sleep": PMDATA_SLEEP}

    print("PMData physiology features:", PMDATA_PHYS)
    print("PMData sleep features:     ", PMDATA_SLEEP)
    print("PMData wellness/target cols:", PMDATA_WELLNESS)
    print("Total NDI features:        ", len(PMDATA_FEATURE_NAMES))

    if not PMDATA_WELLNESS:
        # Try re-loading wellness directly in case Cell 5 merge failed
        import os, glob
        print("\nWellness not found in loaded data — attempting direct re-load...")
        for p in pmdata_participants:
            well_path = os.path.join(
                PMDATA_DIR, p["pid"], "pmsys", "wellness.csv")
            if os.path.isfile(well_path):
                try:
                    w = pd.read_csv(well_path)
                    w.columns = [c.strip().lower().replace(" ","_")
                                 for c in w.columns]
                    if "effective_time_frame" in w.columns:

                        # Convert wellness timestamp to date only
                        w["date"] = pd.to_datetime(
                            w["effective_time_frame"],
                            errors="coerce"
                        ).dt.strftime("%Y-%m-%d")

                        df = p["days_df"].copy()

                        df["date"] = pd.to_datetime(
                            df["date"],
                            errors="coerce"
                        ).dt.strftime("%Y-%m-%d")

                        new_wcols = [
                            c for c in w.columns
                            if c not in ["effective_time_frame", "date"]
                            and c not in df.columns
                        ]

                        df = df.merge(
                            w[["date"] + new_wcols],
                            on="date",
                            how="left"
                        )

                        p["days_df"] = df
                        sample_cols.update(new_wcols)
                except Exception as e:
                    print(f"  wellness re-load {p['pid']}: {e}")

        PMDATA_WELLNESS = [c for c in [
            "fatigue","mood","readiness","sleep_quality","stress","soreness"
        ] if c in sample_cols]
        print("Wellness cols after re-load:", PMDATA_WELLNESS)

    # Build neutrosophic features for each participant
    print("\nComputing neutrosophic features for PMData participants...")
    pmdata_frames = []
    for p in pmdata_participants:
        try:
            df = p["days_df"].copy().reset_index(drop=True)

            for col in PMDATA_FEATURE_NAMES:
                if col not in df.columns:
                    df[col] = np.nan
                df[col] = pd.to_numeric(df[col], errors="coerce")
                df[col] = df[col].fillna(df[col].median())

            X        = df[PMDATA_FEATURE_NAMES].values
            train_mask_full = np.ones(len(X), dtype=bool)
            neutro   = ndi.build_neutrosophic_features(X, PMDATA_FEATURE_NAMES,
                                            train_mask=train_mask_full)
            result   = pd.concat([df[["date"]].reset_index(drop=True),
                                   neutro.reset_index(drop=True)], axis=1)
            result["pid"]    = p["pid"]
            result["max_hr"] = p["max_hr"]

            # Keep raw features AND wellness columns
            for col in PMDATA_FEATURE_NAMES + PMDATA_WELLNESS:
                if col in df.columns:
                    result[col] = df[col].values

            pmdata_frames.append(result)
            p["neutro_df"] = result

            well_coverage = {c: df[c].notna().sum() for c in PMDATA_WELLNESS
                             if c in df.columns}
            print(f"  {p['pid']}: {len(result)} days  "
                  f"NDI={result['NDI'].mean():.4f}  "
                  f"wellness={well_coverage}")

        except Exception as e:
            print(f"  ERROR {p['pid']}: {e}")

    if pmdata_frames:
        pmdata_all = pd.concat(pmdata_frames, ignore_index=True)
        print(f"\nPMData total rows: {len(pmdata_all)}")
        print(f"NDI range: [{pmdata_all['NDI'].min():.4f},"
              f"{pmdata_all['NDI'].max():.4f}]")
        for wcol in PMDATA_WELLNESS:
            if wcol in pmdata_all.columns:
                vals = pmdata_all[wcol].dropna()
                if len(vals) > 0:
                    print(f"  {wcol}: mean={vals.mean():.2f}  "
                          f"range=[{vals.min():.0f},{vals.max():.0f}]  "
                          f"n={len(vals)}")
    else:
        print("No PMData frames built.")
        pmdata_all           = pd.DataFrame()
        PMDATA_FEATURE_NAMES = []
        PMDATA_DOMAIN_MAP    = {}
        PMDATA_WELLNESS      = []


In [ ]:
# D-1  Duplicate-day diagnostic (read-only; writes only to results_diagnostics/)
import os, json
import numpy as np, pandas as pd
_need = ["pmdata_participants", "PMDATA_FEATURE_NAMES", "PMDATA_DIR", "RESULTS_DIR"]
_miss = [g for g in _need if g not in globals()]
assert not _miss, f"Run CELL 1.0, CELL 2, CELL 5 and CELL 6 first. Missing: {_miss}"
LOCAL_TZ = "Europe/Oslo"            # PMData was collected in Norway; Fitbit daily files use local calendar days
DIAG_DIR = os.path.join(os.path.dirname(RESULTS_DIR), "results_diagnostics")
os.makedirs(DIAG_DIR, exist_ok=True)

def _report_dates(w):
    """UTC date (what CELL 6 used) and local calendar date of each wellness report."""
    ts = pd.to_datetime(w["effective_time_frame"], errors="coerce", utc=True)
    raw = w["effective_time_frame"].astype(str)
    has_tz = raw.str.contains(r"(?:Z|[+-]\d\d:?\d\d)$", regex=True)
    utc_date = ts.dt.strftime("%Y-%m-%d")
    local_date = ts.dt.tz_convert(LOCAL_TZ).dt.strftime("%Y-%m-%d").where(has_tz, utc_date)
    return utc_date, local_date, bool(has_tz.all())

summary, detail = [], []
for p in pmdata_participants:
    d = p["days_df"].reset_index(drop=True)
    dates = pd.to_datetime(d["date"], errors="coerce")
    dup_mask = dates.duplicated(keep=False)
    dup_dates = sorted(dates[dup_mask].dt.strftime("%Y-%m-%d").unique())
    feats = d[[c for c in PMDATA_FEATURE_NAMES if c in d.columns]].apply(pd.to_numeric, errors="coerce")
    same_features = all(feats[dates == pd.Timestamp(x)].nunique(dropna=False).max() <= 1 for x in dup_dates)
    uniq = dates.drop_duplicates().sort_values()
    gaps_after_dedup = int((uniq.diff().dt.days.dropna() > 1).sum())
    w_path = os.path.join(PMDATA_DIR, p["pid"], "pmsys", "wellness.csv")
    n_reports = n_shift = 0; tz_aware = None; local_dups = None
    if os.path.isfile(w_path):
        w = pd.read_csv(w_path); w.columns = [c.strip().lower().replace(" ", "_") for c in w.columns]
        if "effective_time_frame" in w.columns:
            utc_date, local_date, tz_aware = _report_dates(w)
            n_reports = int(utc_date.notna().sum())
            n_shift = int((utc_date != local_date).sum())
            local_dups = int(local_date.duplicated().sum())
            for x in dup_dates:
                hit = w[utc_date == x]
                for _, r in hit.iterrows():
                    detail.append(dict(pid=p["pid"], duplicated_date=x, report_timestamp=r["effective_time_frame"],
                                       local_date=local_date.loc[_], mood=r.get("mood", np.nan), stress=r.get("stress", np.nan)))
    summary.append(dict(pid=p["pid"], rows=len(d), unique_dates=int(dates.nunique()), duplicated_dates=len(dup_dates),
                        extra_rows=len(d) - int(dates.nunique()), duplicate_rows_have_identical_features=same_features,
                        calendar_gaps_after_dedup=gaps_after_dedup, wellness_reports=n_reports,
                        reports_whose_utc_date_differs_from_local_date=n_shift,
                        dates_with_2plus_reports_local=local_dups, timestamps_carry_timezone=tz_aware))
summ = pd.DataFrame(summary); det = pd.DataFrame(detail)
summ.to_csv(os.path.join(DIAG_DIR, "duplicate_days_summary.csv"), index=False)
det.to_csv(os.path.join(DIAG_DIR, "duplicate_days_reports.csv"), index=False)
pd.set_option("display.width", 250)
print(summ.to_string(index=False))
print("\nReports behind each duplicated date:"); print(det.to_string(index=False) if len(det) else "none")
print(f"\nTOTAL extra rows: {int(summ['extra_rows'].sum())} across {int((summ['extra_rows'] > 0).sum())} participants; "
      f"calendar gaps after removing duplicates: {int(summ['calendar_gaps_after_dedup'].sum())}")
